<a href="https://colab.research.google.com/github/iantonyjohn/Map-making/blob/main/automated_report_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 0: Install necessary packages
!pip install -q earthengine-api geemap geopandas statsmodels contextily matplotlib-scalebar rasterio

In [3]:
# Cell 1: Import packages, authenticate GEE, and set aesthetics
import os
import glob
import time
import sqlite3
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import statsmodels.api as sm
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib_scalebar.scalebar import ScaleBar
import contextily as ctx
import rasterio
from dateutil.relativedelta import relativedelta
import warnings
import ee
import geemap

warnings.filterwarnings('ignore')

# Authenticate and Initialize Earth Engine
try:
    ee.Initialize(project='ee-antonyjohniirs') # Replace if you know your ID, otherwise it uses default
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
print("✅ Earth Engine Initialized Successfully.")

✅ Earth Engine Initialized Successfully.


In [5]:
# Cell 2: Declare paths and load base data
polygons_dir = "/content/drive/MyDrive/polygons"
samplingdates_path = "/content/drive/MyDrive/sample_ids.csv"
output_location = "/content/drive/MyDrive/IndocertRdFinalReportFigures"

os.makedirs(output_location, exist_ok=True)

try:
    sampling_df = pd.read_csv(samplingdates_path)
    sampling_df.columns = sampling_df.columns.str.strip()
    sampling_df['sampling_date'] = pd.to_datetime(sampling_df['Date of Sampling 2025'], format='%d-%m-%Y', errors='coerce')
    sampling_dict = dict(zip(sampling_df['SAMPLE_ID'].astype(str).str.strip(), sampling_df['sampling_date']))
    print(f"✅ Sampling dates loaded. Found {len(sampling_dict)} valid dates.")
except Exception as e:
    print(f"⚠️ Could not load sampling dates from {samplingdates_path}. Error: {e}")
    sampling_dict = {}

geojson_files = glob.glob(os.path.join(polygons_dir, "*.geojson"))
print(f"📁 Found {len(geojson_files)} GeoJSON files to process.")

✅ Sampling dates loaded. Found 21 valid dates.
📁 Found 20 GeoJSON files to process.


In [6]:
# Cell 3: Generate Quarterly GEE Timeseries with SQLite Storage & R-Style Aesthetics

import os
import sqlite3
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
import warnings
import ee
import geemap
import json  # Added to robustly parse the geometry

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
YEARS = range(2016, 2027)
QUARTERS = {
    'Q1': ('01-01', '03-31', '02-15'),
    'Q2': ('04-01', '06-30', '05-15'),
    'Q3': ('07-01', '09-30', '08-15'),
    'Q4': ('10-01', '12-31', '11-15')
}
db_path = os.path.join(output_location, "farm_metadata.sqlite")
raster_dir = "/content/rasters"
os.makedirs(raster_dir, exist_ok=True)

# --- DATABASE SETUP ---
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS quarterly_data (
        farm_id TEXT, year INTEGER, quarter TEXT, date TEXT,
        ndvi_mean REAL, ndmi_mean REAL, lst_mean REAL,
        ndvi_path TEXT, ndmi_path TEXT, lst_path TEXT,
        UNIQUE(farm_id, year, quarter)
    )
''')
conn.commit()

# --- AESTHETIC PLOTTING (GGPLOT2 STYLE) ---
def plot_aesthetic_timeseries(df, date_col, value_col, index_name, output_path):
    df_valid = df.dropna(subset=[value_col]).sort_values(date_col)
    if df_valid.empty: return

    x_num = mdates.date2num(df_valid[date_col])
    y_vals = df_valid[value_col].values
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

    if len(df_valid) >= 4:
        lowess = sm.nonparametric.lowess(y_vals, x_num, frac=0.25)
        x_smooth, y_smooth = lowess[:, 0], lowess[:, 1]
        window_size = max(len(y_vals) // 8, 3)
        rolling_std = df_valid[value_col].rolling(window=window_size, center=True, min_periods=2).std().fillna(0.05)
        std_smooth = sm.nonparametric.lowess(rolling_std, x_num, frac=0.3)[:, 1]

        ax.fill_between(x_smooth, y_smooth - std_smooth*0.5, y_smooth + std_smooth*0.5, color='#cdd1d5', alpha=0.5, edgecolor='none', zorder=1)
        ax.plot(x_smooth, y_smooth, color='#a0adb9', linewidth=3.5, zorder=2)

    ax.scatter(x_num, y_vals, color='#444444', alpha=0.6, edgecolor='black', linewidth=0.7, s=45, zorder=3)

    if index_name in ['NDVI', 'NDMI']:
        ax.set_ylim(-1.0, 1.0)
        ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
        ax.set_yticklabels(['-1.00', '-0.50', '0.00', '0.50', '1.00'])
    else:
        ax.set_ylim(min(y_vals) - 5, max(y_vals) + 5)

    ax.set_ylabel(index_name, fontsize=16, color='black', labelpad=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(fontsize=13, color='#333333')
    plt.yticks(fontsize=13, color='#333333')

    ax.set_facecolor('#ffffff')
    ax.grid(True, which='major', color='#ebebeb', linestyle='-', linewidth=1.2, zorder=0)
    ax.minorticks_on()
    ax.grid(True, which='minor', color='#f4f4f4', linestyle='-', linewidth=0.7, zorder=0)
    ax.tick_params(which='both', direction='out', length=5, color='#333333')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.close(fig)

# --- GEE SERVER-SIDE PROCESSING ---
def get_gee_quarterly_data(geom, year, q_start, q_end, farm_id, q_name):
    start = f"{year}-{q_start}"
    end = f"{year}-{q_end}"
    results = {'ndvi_mean': None, 'ndmi_mean': None, 'lst_mean': None, 'ndvi_path': None, 'ndmi_path': None, 'lst_path': None}

    # 1. Sentinel-2 NDVI/NDMI
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(geom).filterDate(start, end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))

    def mask_clouds(img):
        scl = img.select('SCL')
        mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
        return img.updateMask(mask)

    if s2.size().getInfo() > 0:
        s2_clean = s2.map(mask_clouds)
        med = s2_clean.select(['B4', 'B8', 'B8A', 'B11']).median().clip(geom)

        ndvi = med.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndmi = med.normalizedDifference(['B8A', 'B11']).rename('NDMI')

        mean_dict = ndvi.addBands(ndmi).reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=10, maxPixels=1e9).getInfo()
        results['ndvi_mean'] = mean_dict.get('NDVI')
        results['ndmi_mean'] = mean_dict.get('NDMI')

        if results['ndvi_mean'] is not None:
            results['ndvi_path'] = os.path.join(raster_dir, f"{farm_id}_NDVI_{year}_{q_name}.tif")
            geemap.ee_export_image(ndvi, filename=results['ndvi_path'], scale=10, region=geom, crs='EPSG:3857')

            results['ndmi_path'] = os.path.join(raster_dir, f"{farm_id}_NDMI_{year}_{q_name}.tif")
            geemap.ee_export_image(ndmi, filename=results['ndmi_path'], scale=10, region=geom, crs='EPSG:3857')

    # 2. Landsat 8/9 LST
    ls8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(geom).filterDate(start, end)
    ls9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(geom).filterDate(start, end)
    ls = ls8.merge(ls9).filter(ee.Filter.lt('CLOUD_COVER', 40))

    if ls.size().getInfo() > 0:
        def apply_scale(img):
            lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
            qa = img.select('QA_PIXEL')
            mask = qa.bitwiseAnd(1 << 3).eq(0)
            return lst.updateMask(mask)

        lst_med = ls.map(apply_scale).median().clip(geom)
        mean_lst = lst_med.reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=30, maxPixels=1e9).getInfo()
        results['lst_mean'] = mean_lst.get('LST')

        if results['lst_mean'] is not None:
            results['lst_path'] = os.path.join(raster_dir, f"{farm_id}_LST_{year}_{q_name}.tif")
            geemap.ee_export_image(lst_med, filename=results['lst_path'], scale=30, region=geom, crs='EPSG:3857')

    return results

# --- EXECUTION LOOP ---
for file_path in geojson_files:
    farm_id = os.path.basename(file_path).replace('.geojson', '')
    farm_output_dir = os.path.join(output_location, farm_id)
    os.makedirs(farm_output_dir, exist_ok=True)

    # FIXED: Robust GeoJSON to EE Geometry conversion using GeoPandas and json
    gdf = gpd.read_file(file_path).to_crs("EPSG:4326")
    geom_dict = json.loads(gdf.to_json())['features'][0]['geometry']
    ee_geom = ee.Geometry(geom_dict)

    print(f"\n🚀 Processing GEE Data for Farm ID: {farm_id}")

    for year in YEARS:
        for q_name, (q_start, q_end, q_rep_date) in QUARTERS.items():
            rep_date = f"{year}-{q_rep_date}"
            res = get_gee_quarterly_data(ee_geom, year, q_start, q_end, farm_id, q_name)

            cursor.execute('''
                INSERT OR REPLACE INTO quarterly_data
                (farm_id, year, quarter, date, ndvi_mean, ndmi_mean, lst_mean, ndvi_path, ndmi_path, lst_path)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (farm_id, year, q_name, rep_date, res['ndvi_mean'], res['ndmi_mean'], res['lst_mean'], res['ndvi_path'], res['ndmi_path'], res['lst_path']))
            conn.commit()

    # Retrieve and Plot
    df = pd.read_sql_query(f"SELECT date, ndvi_mean, ndmi_mean, lst_mean FROM quarterly_data WHERE farm_id = '{farm_id}'", conn)
    df['date'] = pd.to_datetime(df['date'])

    plot_aesthetic_timeseries(df, 'date', 'ndvi_mean', 'NDVI', os.path.join(farm_output_dir, f"{farm_id}_NDVI_TS.png"))
    plot_aesthetic_timeseries(df, 'date', 'ndmi_mean', 'NDMI', os.path.join(farm_output_dir, f"{farm_id}_NDMI_TS.png"))
    plot_aesthetic_timeseries(df, 'date', 'lst_mean', 'LST (Celsius)', os.path.join(farm_output_dir, f"{farm_id}_LST_TS.png"))

conn.close()
print("\n✅ All GEE timeseries extraction complete!")

Streaming output truncated to the last 5000 lines.
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDMI_2021_Q1.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_LST_2021_Q1.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDVI_2021_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDMI_2021_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_LST_2021_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDVI_2021_Q3.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDMI_2021_Q3.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_LST_2021_Q3.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDVI_2021_Q4.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/BCKK-L_NDMI_2021_Q4.tif
Ge

Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_LST_2021_Q3.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDVI_2021_Q4.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDMI_2021_Q4.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_LST_2021_Q4.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDVI_2022_Q1.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDMI_2022_Q1.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_LST_2022_Q1.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDVI_2022_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_NDMI_2022_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/rasters/KCLCP_LST_2022_Q2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/r

In [7]:
# Cell 4: Generate Dense Raster Layout PDFs (RAM Optimized)

import os
import gc  # Added for manual Garbage Collection
import sqlite3
import pandas as pd
import geopandas as gpd
import numpy as np

# MUST BE SET BEFORE IMPORTING PYPLOT to prevent memory caching
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib_scalebar.scalebar import ScaleBar
import contextily as ctx
import rasterio

def build_pdf_from_db(farm_id, gdf, index_col, path_col, output_dir, cmap, vmin, vmax, title):
    conn = sqlite3.connect(os.path.join(output_location, "farm_metadata.sqlite"))
    query = f"SELECT year, quarter, {path_col} FROM quarterly_data WHERE farm_id = '{farm_id}' AND {path_col} IS NOT NULL ORDER BY year, quarter"
    records = pd.read_sql_query(query, conn)
    conn.close()

    if records.empty:
        print(f"⚠️ No database records found for {farm_id} {title}. Skipping PDF.")
        return

    COLS, ROWS = 4, 5
    PLOTS_PER_PAGE = COLS * ROWS
    pdf_path = os.path.join(output_dir, f"{farm_id}_{title}_Dense.pdf")

    # Ensure Web Mercator (EPSG:3857) to match the rasters
    gdf_3857 = gdf.to_crs("EPSG:3857")

    # Filter out 'Points' so ONLY the boundary polygon is drawn
    gdf_polys = gdf_3857[gdf_3857.geometry.type.isin(['Polygon', 'MultiPolygon'])]

    with PdfPages(pdf_path) as pdf:
        for page_idx in range(0, len(records), PLOTS_PER_PAGE):
            page_data = records.iloc[page_idx:page_idx + PLOTS_PER_PAGE]
            fig, axes = plt.subplots(nrows=ROWS, ncols=COLS, figsize=(8.27, 11.69), dpi=300)
            fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.10, wspace=0.05, hspace=0.25)
            axes_flat = axes.flatten()

            for idx, row in enumerate(page_data.itertuples()):
                ax = axes_flat[idx]
                path = getattr(row, path_col)

                if not os.path.exists(path):
                    print(f"❌ Missing file on disk: {path}")
                    ax.axis('off')
                    continue

                try:
                    with rasterio.open(path) as src:
                        arr = src.read(1)
                        # Fix NoData transparency
                        if src.nodata is not None:
                            arr = np.where(arr == src.nodata, np.nan, arr)
                        elif arr.min() < -100:
                            arr = np.where(arr < -100, np.nan, arr)

                        bounds = src.bounds
                        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

                    # Base placeholder to set the extent
                    ax.imshow(np.zeros_like(arr), extent=extent, alpha=0.0)

                    # Fetch basemap (locked to zoom=16 to prevent resolution crashes)
                    try:
                        ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.Esri.WorldImagery, attribution="", zoom=16)
                    except Exception as e:
                        try:
                            # Fallback to OpenStreetMap if Esri is unreachable
                            ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.OpenStreetMap.Mapnik, attribution="", zoom=16)
                        except Exception as e2:
                            pass

                    # 1. Overlay the Data Raster
                    im = ax.imshow(arr, extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)

                    # 2. OVERLAY THE POLYGON BOUNDARY
                    if not gdf_polys.empty:
                        gdf_polys.plot(ax=ax, facecolor="none", edgecolor="#ff0000", linewidth=1.5, zorder=5)

                    ax.axis('off')
                    ax.text(0.5, -0.10, f"{row.year} {row.quarter}", transform=ax.transAxes, ha='center', fontsize=9, fontweight='bold')

                    if page_idx == 0 and idx == 0:
                        ax.add_artist(ScaleBar(1, "m", location="lower left", scale_loc="bottom", color="white", box_alpha=0.4))

                    # Delete the raster array from RAM explicitly
                    del arr

                except Exception as e:
                    print(f"❌ Plot error for {row.year} {row.quarter}: {e}")
                    ax.axis('off')

            for idx in range(len(page_data), len(axes_flat)):
                axes_flat[idx].axis('off')

            fig.text(0.02, 0.02, f"Source: Google Earth Engine\nIndex: {title}", fontsize=6, color='gray')
            cbar_ax = fig.add_axes([0.35, 0.03, 0.3, 0.015])
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
            cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
            cbar.ax.tick_params(labelsize=6)
            cbar.set_label(f'{title}', fontsize=8, fontweight='bold')

            pdf.savefig(fig, facecolor='white')

            # --- AGGRESSIVE MEMORY CLEANUP ---
            fig.clf()               # Clear all axes and elements from the figure
            plt.close('all')        # Forcibly close all matplotlib states
            gc.collect()            # Tell Python to immediately dump unreferenced memory

ndvi_cmap = mcolors.LinearSegmentedColormap.from_list("ndvi", ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850'])
ndmi_cmap = mcolors.LinearSegmentedColormap.from_list("ndmi", ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e'])

# Execution Loop
for file_path in geojson_files:
    farm_id = os.path.basename(file_path).replace('.geojson', '')
    farm_dir = os.path.join(output_location, farm_id)

    # Load the GeoJSON
    gdf = gpd.read_file(file_path)

    print(f"Building PDFs for Farm ID: {farm_id}...")

    build_pdf_from_db(farm_id, gdf, 'ndvi_mean', 'ndvi_path', farm_dir, ndvi_cmap, -0.2, 0.8, 'NDVI')
    build_pdf_from_db(farm_id, gdf, 'ndmi_mean', 'ndmi_path', farm_dir, ndmi_cmap, -0.4, 0.6, 'NDMI')
    build_pdf_from_db(farm_id, gdf, 'lst_mean', 'lst_path', farm_dir, 'inferno', 15, 45, 'LST')

print("✅ PDF Generation Complete.")

Building PDFs for Farm ID: HML-H...
Building PDFs for Farm ID: HML-L...
Building PDFs for Farm ID: BCKOT-Hl...
Building PDFs for Farm ID: BCKK-L...
Building PDFs for Farm ID: BCKK-H...
Building PDFs for Farm ID: NESK-L...
Building PDFs for Farm ID: NESK-H...
Building PDFs for Farm ID: MCBSTP...
Building PDFs for Farm ID: WAM-T...
Building PDFs for Farm ID: DHARM...
Building PDFs for Farm ID: KCLKB...
Building PDFs for Farm ID: KCLAR...
Building PDFs for Farm ID: MMK-Av...
Building PDFs for Farm ID: POABS-T...
Building PDFs for Farm ID: POABS-C...
Building PDFs for Farm ID: KCLCP...
Building PDFs for Farm ID: HMLVA...
Building PDFs for Farm ID: HMLAR...
Building PDFs for Farm ID: HMLTH...
Building PDFs for Farm ID: BCKOT-Ti...
✅ PDF Generation Complete.


In [10]:
# Cell 5: Rainfall pattern graph using GEE CHIRPS dataset (Fixed Data Types)

import json
import seaborn as sns
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import ee
from dateutil.relativedelta import relativedelta

def plot_rainfall(df, sampling_date, farm_id, output_path):
    # Ensure there's actually data to plot
    if df is None or df.empty or df['precip'].sum() == 0:
        print(f"⚠️ No rainfall data found for {farm_id}. Skipping plot.")
        return

    fig, ax1 = plt.subplots(figsize=(10, 5), dpi=300)

    ax1.bar(df['date'], df['precip'], color='#4a90e2', alpha=0.7, label='Actual Rainfall (mm)', width=20)
    ax1.set_ylabel("Monthly Rainfall (mm)", color='#333333', fontsize=11)

    ax2 = ax1.twinx()
    # Smoothed trend to act as "Normal" representation
    df['normal_precip'] = df['precip'].rolling(window=3, center=True, min_periods=1).mean()
    sns.lineplot(data=df, x='date', y='normal_precip', color='#e74c3c', linewidth=2.5, marker='o', ax=ax2, label='3-Month Trend')
    ax2.set_ylabel("Trend (mm)", color='#e74c3c', fontsize=11)
    ax2.grid(False)

    ax1.axvline(pd.to_datetime(sampling_date), color='#2c3e50', linestyle='--', linewidth=2, zorder=5)
    ax1.text(pd.to_datetime(sampling_date) + relativedelta(days=5), ax1.get_ylim()[1]*0.9, 'Sampling Date', color='#2c3e50', fontweight='bold', rotation=90)

    plt.title(f"Rainfall Pattern 1-Year Window - {farm_id}", fontsize=14, pad=15, fontweight='bold')
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate(rotation=45)

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', frameon=True, facecolor='white')

    ax1.spines['top'].set_visible(False); ax2.spines['top'].set_visible(False)
    ax1.set_facecolor('#ffffff')
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close(fig)

for file_path in geojson_files:
    farm_id = os.path.basename(file_path).replace('.geojson', '')
    farm_dir = os.path.join(output_location, farm_id)

    if farm_id in sampling_dict:
        s_date = sampling_dict[farm_id]
        if pd.isna(s_date): continue

        start_date = (s_date - relativedelta(months=6)).strftime('%Y-%m-%d')
        end_date = (s_date + relativedelta(months=6)).strftime('%Y-%m-%d')
        print(f"Fetching CHIRPS rainfall for {farm_id} ({start_date} to {end_date})...")

        # Load GeoJSON and strictly isolate the Polygon boundary
        gdf = gpd.read_file(file_path).to_crs("EPSG:4326")
        gdf_poly = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]

        poly_geojson = json.loads(gdf_poly.to_json())
        geom = ee.FeatureCollection(poly_geojson).geometry()

        chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(geom).filterDate(start_date, end_date)

        def compute_monthly_sum(image):
            # Use scale=1000 to force an intersection reading for small farms
            val = image.reduceRegion(ee.Reducer.mean(), geom, 1000).get('precipitation')
            return ee.Feature(None, {'date': image.date().format('YYYY-MM'), 'precip': val})

        if chirps.size().getInfo() > 0:
            rain_data = chirps.map(compute_monthly_sum).getInfo()['features']

            df_rain = pd.DataFrame([feat['properties'] for feat in rain_data])

            # Ensure 'precip' exists in case GEE returns empty properties
            if 'precip' not in df_rain.columns:
                df_rain['precip'] = 0.0

            # Strictly force precip column to a float and replace nulls with 0 BEFORE grouping
            df_rain['precip'] = pd.to_numeric(df_rain['precip'], errors='coerce').fillna(0.0)
            df_rain['date'] = pd.to_datetime(df_rain['date'])

            # Explicitly group the 'precip' column so Pandas doesn't try to drop it
            df_monthly = df_rain.groupby('date', as_index=False)['precip'].sum()

            plot_rainfall(df_monthly, s_date, farm_id, os.path.join(farm_dir, f"{farm_id}_Rainfall.png"))

print("✅ Rainfall Processing Complete.")

Fetching CHIRPS rainfall for BCKOT-Hl (2025-06-12 to 2026-06-12)...
Fetching CHIRPS rainfall for BCKK-L (2025-06-12 to 2026-06-12)...
Fetching CHIRPS rainfall for BCKK-H (2025-06-12 to 2026-06-12)...
Fetching CHIRPS rainfall for NESK-L (2025-06-12 to 2026-06-12)...
Fetching CHIRPS rainfall for NESK-H (2025-06-12 to 2026-06-12)...
Fetching CHIRPS rainfall for MCBSTP (2025-06-07 to 2026-06-07)...
Fetching CHIRPS rainfall for WAM-T (2025-06-07 to 2026-06-07)...
Fetching CHIRPS rainfall for DHARM (2025-06-07 to 2026-06-07)...
Fetching CHIRPS rainfall for KCLKB (2025-06-06 to 2026-06-06)...
Fetching CHIRPS rainfall for KCLAR (2025-06-06 to 2026-06-06)...
Fetching CHIRPS rainfall for MMK-Av (2025-05-26 to 2026-05-26)...
Fetching CHIRPS rainfall for POABS-T (2025-05-23 to 2026-05-23)...
Fetching CHIRPS rainfall for POABS-C (2025-05-23 to 2026-05-23)...
Fetching CHIRPS rainfall for KCLCP (2025-06-07 to 2026-06-07)...
Fetching CHIRPS rainfall for HMLVA (2025-06-06 to 2026-06-06)...
Fetching CHI

In [11]:
import shutil
from google.colab import files

print("Zipping the folder. This might take a minute...")
# Creates a file named 'rasters_archive.zip' in the /content folder
shutil.make_archive('/content/rasters_archive', 'zip', '/content/rasters')

print("Zipping complete! Starting download...")
# Triggers the browser prompt to download the file
files.download('/content/rasters_archive.zip')

Zipping the folder. This might take a minute...
Zipping complete! Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>